# Offline blend between run ROW and run HYBRID (no GPU, no retraining)

Combines the OOFs of the two hybrid runs (same folds, same rows) and measures with **nested** blending whether mixing the models of the two runs or averaging them brings a gain beyond noise.

**Input** (two folders, each with `oof_resnet.npy`, `oof_ft_transformer.npy`, `oof_catboost.npy`, `oof_targets.npy`, `cv_fold_of_row.npy`):
- `row_run`: uses hybrid CV protocol, with row-wise-tuned hyperparameters (OOF nested blend 0.01714)
- `hybrid_run`: uses hybrid CV protocol, with hybrid-tuned hyperparameters (OOF nested blend 0.01715)

**Decision rule:** a variant is worth it only if it improves upon hybrid-hybrid baseline by at least `MIN_GAIN` on average **and** on at least `MIN_FOLDS_BETTER` out of 5 folds (paired differences per fold).


In [1]:
# Step 1 - configuration
import os, json
import numpy as np
import pandas as pd
from scipy.optimize import minimize

ROW_RUN = "/kaggle/input/datasets/marcomerlo37/moa-diagnostics-dataset"
HYBRID_RUN = "/kaggle/input/datasets/marcomerlo37/moa-oof-dataset"        
CLIP_EPS = 1e-15
MIN_GAIN = 5e-5
MIN_FOLDS_BETTER = 4
MODEL_FILES = {"resnet": "oof_resnet.npy", "ft": "oof_ft_transformer.npy", "cb": "oof_catboost.npy"}


In [2]:
# Step 2 - load and verify that the two runs are aligned
def load_run(folder):
    return {"Y": np.load(f"{folder}/oof_targets.npy"),
            "fold": np.load(f"{folder}/cv_fold_of_row.npy"),
            **{name: np.load(f"{folder}/{file}") for name, file in MODEL_FILES.items()}}

row_run, hybrid_run = load_run(ROW_RUN), load_run(HYBRID_RUN)
assert row_run["Y"].shape == hybrid_run["Y"].shape and (row_run["Y"] == hybrid_run["Y"]).all(), "different rows/targets between the two runs"
assert (row_run["fold"] == hybrid_run["fold"]).all(), "different folds between the two runs: paired comparison is invalid"
Y, fold_of_row = hybrid_run["Y"], hybrid_run["fold"]
N_FOLDS = int(fold_of_row.max()) + 1
splits = [(np.where(fold_of_row != f)[0], np.where(fold_of_row == f)[0]) for f in range(N_FOLDS)]
print(f"Rows {Y.shape[0]} | targets {Y.shape[1]} | folds {N_FOLDS} | validation sizes {[len(v) for _, v in splits]}")

OOF = {f"{m}_{tag}": run[m] for tag, run in (("ROW", row_run), ("HYBRID", hybrid_run)) for m in MODEL_FILES}
for pair in (("resnet_ROW", "resnet_HYBRID"), ("ft_ROW", "ft_HYBRID"), ("cb_ROW", "cb_HYBRID")):
    OOF[pair[0].split("_")[0] + "_avg"] = 0.5 * (OOF[pair[0]] + OOF[pair[1]])


Rows 21948 | targets 206 | folds 5 | validation sizes [4386, 4356, 4371, 4383, 4452]


In [3]:
# Step 3 - metric and nested blend (same functions as training notebook, but silent and with per-fold score)
def log_loss_mean(y, p, eps=CLIP_EPS):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))

def optimize_weights(oof_list, y):
    def objective(logits):
        w = np.exp(logits - logits.max()); w /= w.sum()
        return log_loss_mean(y, sum(wi * p for wi, p in zip(w, oof_list)))
    result = minimize(objective, x0=np.zeros(len(oof_list)), method="Nelder-Mead",
                      options={"xatol": 1e-6, "fatol": 1e-9, "maxiter": 3000})
    w = np.exp(result.x - result.x.max())
    return w / w.sum()

def nested_blend(names):
    """Weights optimized on training folds only, score on left-out fold. Returns per-fold scores and average weights."""
    oof_list = [OOF[n] for n in names]
    scores, weights = [], []
    for train_idx, val_idx in splits:
        w = optimize_weights([p[train_idx] for p in oof_list], Y[train_idx])
        scores.append(log_loss_mean(Y[val_idx], sum(wi * p[val_idx] for wi, p in zip(w, oof_list))))
        weights.append(w)
    sizes = [len(v) for _, v in splits]
    return np.array(scores), float(np.average(scores, weights=sizes)), np.mean(weights, axis=0)


In [ ]:
# Step 4 - variants to compare
VARIANTS = {
    "ROW (all run ROW)":                             ["resnet_ROW", "ft_ROW", "cb_ROW"],
    "HYBRID (baseline, all run HYBRID)":             ["resnet_HYBRID", "ft_HYBRID", "cb_HYBRID"],
    "ResNet HYBRID + FT ROW + CatBoost HYBRID":      ["resnet_HYBRID", "ft_ROW", "cb_HYBRID"],
    "ResNet HYBRID + FT ROW + CatBoost ROW":         ["resnet_HYBRID", "ft_ROW", "cb_ROW"],
    "ResNet avg + FT avg + CB avg":                  ["resnet_avg", "ft_avg", "cb_avg"],
    "6 models, free weights":                        ["resnet_ROW", "resnet_HYBRID", "ft_ROW", 
                                                      "ft_HYBRID", "cb_ROW", "cb_HYBRID"],
}
results = {}
for label, names in VARIANTS.items():
    fold_scores, mean_score, mean_weights = nested_blend(names)
    results[label] = {"names": names, "fold_scores": fold_scores, "score": mean_score, "weights": mean_weights}
    print(f"{label:38s} nested {mean_score:.5f}")

# sanity check: baselines must reproduce numbers from training notebooks (ROW 0.01714, HYBRID 0.01715)
print("\nSanity check: expected ROW ~0.01714, expected HYBRID ~0.01715")


ROW (all run ROW)                      nested 0.01720
HYBRID (baseline, all run HYBRID)      nested 0.01715
ResNet HYBRID + FT ROW + CatBoost HYBRID nested 0.01715
ResNet HYBRID + FT ROW + CatBoost ROW  nested 0.01715
ResNet avg + FT avg + CB avg           nested 0.01715
6 models, free weights                 nested 0.01714

Sanity check: expected ROW ~0.01714, expected HYBRID ~0.01715


In [5]:
# Step 5 - paired comparison with baseline HYBRID
base = results["HYBRID (baseline, all run HYBRID)"]
rows = []
for label, res in results.items():
    diff = res["fold_scores"] - base["fold_scores"]          # negative = better than baseline
    rows.append({"variant": label, "nested": res["score"], "delta_vs_HYBRID": res["score"] - base["score"],
                 "folds_better": int((diff < 0).sum()), "delta_per_fold(1e-5)": np.round(diff * 1e5, 0).astype(int).tolist(),
                 "adoptable": bool(label != "HYBRID (baseline, all run HYBRID)" and -(res["score"] - base["score"]) >= MIN_GAIN
                                   and (diff < 0).sum() >= MIN_FOLDS_BETTER)})
report = pd.DataFrame(rows)
print(report.round(5).to_string(index=False))
print("\nAverage weights (order of models in variant):")
for label, res in results.items():
    print(f"  {label:38s} " + ", ".join(f"{n}={w:.3f}" for n, w in zip(res["names"], res["weights"])))
print(f"\nThreshold: gain >= {MIN_GAIN} and better on >= {MIN_FOLDS_BETTER}/{N_FOLDS} folds. "
      "Variants are chosen on the same OOF: gains close to the threshold should be treated as noise.")


                                 variant  nested  delta_vs_HYBRID  folds_better delta_per_fold(1e-5)  adoptable
                       ROW (all run ROW) 0.01720          0.00005             0     [5, 1, 3, 7, 10]      False
       HYBRID (baseline, all run HYBRID) 0.01715          0.00000             0      [0, 0, 0, 0, 0]      False
ResNet HYBRID + FT ROW + CatBoost HYBRID 0.01715         -0.00000             3    [-1, 0, -2, 0, 2]      False
   ResNet HYBRID + FT ROW + CatBoost ROW 0.01715          0.00000             3    [-1, 0, -2, 2, 3]      False
            ResNet avg + FT avg + CB avg 0.01715          0.00000             2     [0, -2, 0, 2, 2]      False
                  6 models, free weights 0.01714         -0.00000             4   [-2, -1, -1, 0, 2]      False

Average weights (order of models in variant):
  ROW (all run ROW)                      resnet_ROW=0.298, ft_ROW=0.485, cb_ROW=0.217
  HYBRID (baseline, all run HYBRID)      resnet_HYBRID=0.478, ft_HYBRID=0.347, cb_H